<a href="https://colab.research.google.com/github/abhash-subedi25/nn/blob/main/sleep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [70]:
import pandas as pd
import torch
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.optim as optim

In [50]:
df = pd.read_csv('sleep.csv')
df.head()

,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,NaN
1,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
2,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
3,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
4,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea


In [62]:
non_numeric = df.select_dtypes(exclude='number').columns.tolist()
non_numeric


[]

ValueError: invalid literal for int() with base 10: 'Insomnia'

In [61]:
df['Sleep Disorder'] = df['Sleep Disorder'].fillna(0)
choices = {'Insomnia':2, 0:0, 1:1}
df['Sleep Disorder'] = df['Sleep Disorder'].map(choices)
df.head()

,Person ID,Gender,Age,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,Sleep Disorder
0,1,0,27,6.1,6,42,6,1,77,4200,0
1,2,0,28,6.2,6,60,8,0,75,10000,0
2,3,0,28,6.2,6,60,8,0,75,10000,0
3,4,0,28,5.9,4,30,8,2,85,3000,1
4,5,0,28,5.9,4,30,8,2,85,3000,1


In [56]:
df['Gender'] = np.where(df['Gender'].str.strip() == 'Male', 0,1)
df.head()

,Person ID,Gender,Age,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,Sleep Disorder
0,1,0,27,6.1,6,42,6,1,77,4200,0
1,2,0,28,6.2,6,60,8,0,75,10000,0
2,3,0,28,6.2,6,60,8,0,75,10000,0
3,4,0,28,5.9,4,30,8,2,85,3000,1
4,5,0,28,5.9,4,30,8,2,85,3000,1


In [16]:
print(df['BMI Category'].unique())

['Overweight' 'Normal' 'Obese' 'Normal Weight']


In [25]:
df.columns = df.columns.str.strip()

In [53]:
catagories = {'Normal':0,'Normal Weight':0, 'Overweight':1, 'Obese':2}
df['BMI Category'] = df['BMI Category'].astype(str).str.strip().map(catagories)
df.head()

,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,Software Engineer,6.1,6,42,6,1,126/83,77,4200,0
1,2,Male,28,Doctor,6.2,6,60,8,0,125/80,75,10000,0
2,3,Male,28,Doctor,6.2,6,60,8,0,125/80,75,10000,0
3,4,Male,28,Sales Representative,5.9,4,30,8,2,140/90,85,3000,1
4,5,Male,28,Sales Representative,5.9,4,30,8,2,140/90,85,3000,1


In [35]:
print(df['Blood Pressure'].unique())

['126/83' '125/80' '140/90' '120/80' '132/87' '130/86' '117/76' '118/76'
 '128/85' '131/86' '128/84' '115/75' '135/88' '129/84' '130/85' '115/78'
 '119/77' '121/79' '125/82' '135/90' '122/80' '142/92' '140/95' '139/91'
 '118/75']


In [54]:
df = df.drop(columns=['Occupation', 'Blood Pressure'])
df.head()

,Person ID,Gender,Age,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,6.1,6,42,6,1,77,4200,0
1,2,Male,28,6.2,6,60,8,0,75,10000,0
2,3,Male,28,6.2,6,60,8,0,75,10000,0
3,4,Male,28,5.9,4,30,8,2,85,3000,1
4,5,Male,28,5.9,4,30,8,2,85,3000,1


In [65]:
X = torch.tensor(df.drop('Sleep Disorder', axis=1).values).float()
y = torch.tensor(df['Sleep Disorder'].values).float()


In [80]:
class neu(nn.Module):
  def __init__(self, ins,hs,os):
    super().__init__()
    self.layers = nn.Sequential(
      nn.Linear(ins,hs), nn.ReLU(),
      nn.Linear(hs,hs), nn.ReLU(),
      nn.Linear(hs,os), nn.Sigmoid()
    )

  def forward(self, X):
    return self.layers(X)


input_s = X.shape[1]
hidden_s = 16
output_s = 374

model = neu(input_s, hidden_s, output_s)

criterion = nn.MSELoss()
opt = optim.SGD(model.parameters(), lr=0.01)

epoch = 1000

for epoch in range(epoch):
  y_pred = model(X)
  loss = criterion(y_pred.squeeze(), y)

  opt.zero_grad()
  loss.backward()
  opt.step()

  if (epoch+1) % 10 == 0:
    print(f'epoch: {epoch+1}, loss = {loss.item():.4f}')


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/loss.py:630: UserWarning: Using a target size (torch.Size([374])) that is different to the input size (torch.Size([374, 374])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


epoch: 10, loss = 0.8707
epoch: 20, loss = 0.8469
epoch: 30, loss = 0.8183
epoch: 40, loss = 0.8182
epoch: 50, loss = 0.8182
epoch: 60, loss = 0.8182
epoch: 70, loss = 0.8182
epoch: 80, loss = 0.8182
epoch: 90, loss = 0.8182
epoch: 100, loss = 0.8182
epoch: 110, loss = 0.8182
epoch: 120, loss = 0.8182
epoch: 130, loss = 0.8182
epoch: 140, loss = 0.8182
epoch: 150, loss = 0.8182
epoch: 160, loss = 0.8182
epoch: 170, loss = 0.8182
epoch: 180, loss = 0.8182
epoch: 190, loss = 0.8182
epoch: 200, loss = 0.8182
epoch: 210, loss = 0.8182
epoch: 220, loss = 0.8182
epoch: 230, loss = 0.8182
epoch: 240, loss = 0.8182
epoch: 250, loss = 0.8182
epoch: 260, loss = 0.8182
epoch: 270, loss = 0.8182
epoch: 280, loss = 0.8182
epoch: 290, loss = 0.8182
epoch: 300, loss = 0.8182
epoch: 310, loss = 0.8182
epoch: 320, loss = 0.8182
epoch: 330, loss = 0.8182
epoch: 340, loss = 0.8182
epoch: 350, loss = 0.8182
epoch: 360, loss = 0.8182
epoch: 370, loss = 0.8182
epoch: 380, loss = 0.8182
epoch: 390, loss = 0.

In [77]:
# 1. Ensure y is 1D and matches the number of samples in X
print("X shape:", X.shape)  # Should be (374, num_features)
print("y shape before:", y.shape)

# If y has the wrong shape or needs to be a float column:
y = y.view(-1).float()  # Flatten to match batch size
print("y shape after:", y.shape)  # Should be (374,)

X shape: torch.Size([374, 10])
y shape before: torch.Size([374])
y shape after: torch.Size([374])
